# MCP version of the applicant application

The mcp servers use Rapid api to get linkedin posts, find the relevant posts, send an email to the user for the selected posts, with a curated cover letter and curated c.v


In [1]:
# imports
import os
import requests

from applicant import Applicant
from schema import (
    JobPosts,
)
from agents import Agent, Runner, trace
from agents.mcp import MCPServerSse
from templates import listing_instructions
from dotenv import load_dotenv
from pypdf import PdfReader
from pathlib import Path
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
# set up profile
name = "Denis Gathondu"
denis = Applicant.get(name)
BASE_DIR = Path.cwd()
profile_pdf_path = os.path.abspath(os.path.join(BASE_DIR, "sandbox", "linkedin.pdf"))
with open(profile_pdf_path, "rb") as pdf_file:
    reader = PdfReader(pdf_file)
    profile = "\n".join([page.extract_text() for page in reader.pages])
denis.set_summary(profile)


In [ ]:
rapid_api_key = os.getenv("RAPID_API_KEY")
rapid_api_host = os.getenv("RAPID_API_HOST")

url = f"https://{rapid_api_host}/active-jb-7d"

querystring = {
    "limit": "10",
    "offset": "0",
    "advanced_title_filter": "'AI Engineer' | Software:*",
    "location_filter": '"Nairobi"',
    "description_type": "text",
}

headers = {
    "x-rapidapi-key": rapid_api_key,
    "x-rapidapi-host": rapid_api_host,
    "Content-Type": "application/json",
}

response = requests.get(url, headers=headers, params=querystring)


3


In [6]:
listing_message = f"""
Given the following profile and linked in results. Find me the relevant job posts for {name}.

profile: {denis}
linkedin_results: {response.json()}
"""

params = {"url": "http://127.0.0.1:8000/sse"}

async with MCPServerSse(params, client_session_timeout_seconds=30) as listing_server:
    listing_agent = Agent(
        name="listing_agent",
        instructions=listing_instructions(name),
        model="gpt-4o-mini",
        mcp_servers=[listing_server],
    )
    with trace("Listing Agent"):
        result = await Runner.run(listing_agent, listing_message)
    display(Markdown(result.final_output))


The following job posts have been processed for Denis Gathondu:

1. **Software Engineer - Cloud Images**
   - **Company:** Canonical
   - **Location:** Remote
   - **Salary Range:** Not specified
   - **Job Description:** Canonical seeks a Software Engineer for Linux and cloud infrastructure work, focusing on automation and open source. The role involves collaboration with cloud partners to enhance Ubuntu's capabilities. Ideal candidates should have experience in Python, cloud technologies, and a background in software development.
   - **Job Requirements:** Build and maintain automated pipelines, collaborate with distributed teams, develop features for Ubuntu, debug issues, and write quality code.
   - **Technologies Needed:** Python, Jenkins, cloud platforms, Linux
   - **Must Have Skills:** Software development skills, automation, team collaboration, high code quality
   - **Link to Job Posting:** [Canonical Job Posting](https://ke.linkedin.com/jobs/view/software-engineer-cloud-images-at-canonical-4391513448)
   - **Job Post Date:** 2026-03-27

2. **Head of Software Engineering**
   - **Company:** Nameless Ventures
   - **Location:** Nairobi, Nairobi County, Kenya
   - **Salary Range:** 1m - 1.5m KES per month
   - **Job Description:** Nameless Ventures is hiring a Head of Engineering to manage product development and team integration post-merger. This role requires leadership in software engineering, particularly in a backend setting, to drive the creation of diverse digital products.
   - **Job Requirements:** Oversee engineering efforts in a fintech environment, lead teams, must have backend experience and familiarity with event-driven systems, particularly in a startup setting.
   - **Technologies Needed:** Golang, backend technologies, event-driven systems
   - **Must Have Skills:** Leading engineering teams, overseeing product integration, backend development
   - **Link to Job Posting:** [Nameless Ventures Job Posting](https://ke.linkedin.com/jobs/view/head-of-software-engineering-at-nameless-ventures-4390852472)
   - **Job Post Date:** 2026-03-26

3. **IT Software Services Delivery Manager (Relocation to Mexico)**
   - **Company:** BCS Technology International Pty Ltd
   - **Location:** Nairobi, Nairobi County, Kenya
   - **Salary Range:** Not specified
   - **Job Description:** The IT Software Services Delivery Manager is responsible for overseeing the software delivery process, ensuring projects align with client expectations and are completed efficiently. This role requires excellent management and operational skills in technology projects.
   - **Job Requirements:** Manage delivery of software services, ensure quality, client relationships, lead teams, and improve delivery processes.
   - **Technologies Needed:** Agile, Scrum, SDLC methodologies, cloud technologies
   - **Must Have Skills:** Software delivery management, client management, team leadership, risk management
   - **Link to Job Posting:** [BCS Technology Job Posting](https://ke.linkedin.com/jobs/view/it-software-services-delivery-manager-relocation-to-mexico-at-bcs-technology-international-pty-ltd-4389086341)
   - **Job Post Date:** 2026-03-23

All posts have been successfully added to the database.

JobPosts(job_posts=[JobPost(id=1, title='Senior Software Engineer', company_name='ABC Tech', company_url='https://www.abctech.com', location='Nairobi, Kenya', salary_range='Not specified', job_description='We are looking for a Senior Software Engineer to lead development projects and improve our software solutions. You will collaborate with cross-functional teams to enhance user experiences and deliver high-quality products.', job_requirements='5+ years of software development experience. Proficiency in Python, Django, React, and SQL. Strong understanding of Agile methodologies. Excellent problem-solving skills.', technologies_needed='Python, Django, React, SQL', must_have_skills='Python, Django, React', link_to_job_posting='https://www.linkedin.com/jobs/view/123', job_post_date='2023-09-25'), JobPost(id=2071207383, title='IT Software Services Delivery Manager (Relocation to Mexico)', company_name='BCS Technology International Pty Ltd', company_url='https://www.linkedin.com/company/bcs